# Fase 3: Reentrenamiento Continuo
## CARDIA-RT ISRA

**Autor:** Elias Bejarano Lozada  
**Institucion:** Instituto Tecnologico de Tijuana (ITT)  
**Programa:** Ingenieria Biomedica — Residencia Profesional

---

Esta fase implementa **aprendizaje incremental** para combatir el *concept drift* en la clasificacion de arritmias ECG. En lugar de reentrenar el modelo desde cero cada vez que llegan nuevos datos, los parametros del modelo se actualizan progresivamente con nuevos lotes de latidos ECG mientras se preserva el conocimiento previamente aprendido mediante la tecnica de **replay buffer** (buffer de repeticion).

**Problema:** Los modelos estaticos degradan su rendimiento conforme llegan datos de nuevos pacientes cuya distribucion difiere ligeramente de los datos de entrenamiento original (concept drift).

**Solucion:** Reentrenamiento incremental con replay buffer, que mezcla nuevas muestras con muestras representativas de datos anteriores, evitando la perdida catastrofica del conocimiento.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "scikit-learn", "seaborn", "-q"])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import deque
import json, os, copy

DRIVE_PATH  = "/content/drive/MyDrive/ADAPT-ECG"
DATA_PATH   = DRIVE_PATH + "/data/processed"
MODELS_PATH = DRIVE_PATH + "/models"
DOCS_PATH   = DRIVE_PATH + "/docs"

os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(DOCS_PATH, exist_ok=True)

AAMI_CLASSES = ["N", "S", "V", "F", "Q"]
N_CLASSES    = 5
BEAT_LEN     = 71
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo: {}".format(device))

## Paso 1: Cargar Dataset y Modelo Base

In [ ]:
# ── Definicion de la arquitectura ECG_CNN (identica a Fase 2) ─────────────────
class ECG_CNN(nn.Module):
    """
    Red neuronal convolucional para clasificacion de arritmias ECG.

    Arquitectura: 3 bloques convolucionales (32/64/128 filtros) con
    BatchNorm, ReLU y MaxPool, seguidos de AdaptiveAvgPool, aplanado
    y dos capas densas con Dropout(0.5). Salida log-softmax de 5 clases
    segun el estandar AAMI (N, S, V, F, Q).
    """

    def __init__(self, n_classes: int = 5, beat_len: int = 71):
        super(ECG_CNN, self).__init__()

        # Bloque 1: 1 -> 32 filtros
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )
        # Bloque 2: 32 -> 64 filtros
        self.block2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )
        # Bloque 3: 64 -> 128 filtros
        self.block3 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )

        # Reduccion a longitud fija independiente del input
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()

        # Capas densas
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return nn.functional.log_softmax(x, dim=1)


# ── Cargar datos procesados ────────────────────────────────────────────────────
X = np.load(DATA_PATH + "/X.npy")
y = np.load(DATA_PATH + "/y.npy")

print("Dataset cargado:")
print("  X shape: {}  dtype: {}".format(X.shape, X.dtype))
print("  y shape: {}  dtype: {}".format(y.shape, y.dtype))
print("  Clases presentes: {}".format(np.unique(y)))

# ── Cargar modelo base entrenado en Fase 2 ────────────────────────────────────
base_model = ECG_CNN(n_classes=N_CLASSES, beat_len=BEAT_LEN).to(device)
weights_path = MODELS_PATH + "/ADAPT-ECG-MB.pth"
base_model.load_state_dict(torch.load(weights_path, map_location=device))
base_model.eval()

n_params = sum(p.numel() for p in base_model.parameters())
print("\nModelo base cargado desde: {}".format(weights_path))
print("  Parametros totales: {:,}".format(n_params))

## Paso 2: Simulacion de Concept Drift

Para simular el *concept drift*, dividimos el dataset en **5 lotes secuenciales** que representan la llegada progresiva de datos de nuevos pacientes a lo largo del tiempo. El modelo base fue entrenado en el Lote 0. Los Lotes 1-4 representan pacientes que llegan despues del despliegue clinico.

Primero medimos como se comporta el **modelo estatico** (sin reentrenamiento) en cada lote para demostrar la degradacion del rendimiento. En un escenario real, la distribucion de los latidos puede cambiar debido a diferencias entre pacientes, equipos de adquisicion, condiciones clinicas, etc.

In [ ]:
# Dividir dataset en 5 lotes secuenciales (simula llegada de nuevos pacientes)
N_BATCHES = 5
indices = np.random.RandomState(42).permutation(len(X))
batch_size_drift = len(X) // N_BATCHES
batches = []
for i in range(N_BATCHES):
    start = i * batch_size_drift
    end = start + batch_size_drift if i < N_BATCHES - 1 else len(X)
    batch_idx = indices[start:end]
    batches.append((X[batch_idx], y[batch_idx]))

print("Dataset dividido en {} lotes:".format(N_BATCHES))
for i, (bx, by) in enumerate(batches):
    print("  Lote {}: {:,} latidos".format(i, len(bx)))

In [ ]:
def evaluate_batch(model, X_batch, y_batch, device, batch_size=256):
    """Evalua el modelo en un lote de datos. Retorna accuracy y F1 macro."""
    model.eval()
    X_t = torch.tensor(X_batch, dtype=torch.float32).unsqueeze(1)
    y_t = torch.tensor(y_batch, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=False)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(device))
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(yb.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return acc, f1

static_accuracies = []
static_f1s = []

print("Rendimiento del modelo ESTATICO por lote:")
print("{:>6} | {:>10} | {:>8}".format("Lote", "Accuracy", "F1 Macro"))
print("-" * 30)

for i, (bx, by) in enumerate(batches):
    acc, f1 = evaluate_batch(base_model, bx, by, device)
    static_accuracies.append(acc)
    static_f1s.append(f1)
    print("{:>6} | {:>9.2f}% | {:>8.4f}".format(i, acc*100, f1))

## Paso 3: Modulo de Reentrenamiento Continuo (Replay Buffer)

La tecnica de **replay buffer** (buffer de repeticion) mantiene en memoria un conjunto reducido de muestras de datos anteriores. Cuando llegan nuevos datos, el entrenamiento utiliza tanto las nuevas muestras como una seleccion aleatoria del buffer. Esto logra dos objetivos:

1. **Adaptacion:** el modelo aprende los patrones de los nuevos pacientes.
2. **Retencion:** el modelo no olvida lo aprendido de pacientes anteriores (*evita la perdida catastrofica*).

El tamano del buffer (`max_size`) es un hiperparametro que balancea **estabilidad** (memoria del pasado) vs **plasticidad** (capacidad de adaptarse a lo nuevo). Un buffer mas grande conserva mas conocimiento previo pero incrementa el costo computacional.

In [ ]:
class ReplayBuffer:
    """
    Buffer de repeticion para aprendizaje incremental.

    Almacena muestras de datos anteriores para mezclarlas con nuevos
    datos durante el reentrenamiento, evitando la perdida catastrofica
    del conocimiento previamente adquirido.

    Parameters
    ----------
    max_size : int
        Capacidad maxima del buffer.
    """

    def __init__(self, max_size=2000):
        self.buffer_X = []
        self.buffer_y = []
        self.max_size = max_size

    def add(self, X_batch, y_batch):
        """Agrega nuevas muestras al buffer, descartando las mas antiguas si esta lleno."""
        for x, y in zip(X_batch, y_batch):
            self.buffer_X.append(x)
            self.buffer_y.append(y)

        # Mantener solo las max_size muestras mas recientes
        if len(self.buffer_X) > self.max_size:
            self.buffer_X = self.buffer_X[-self.max_size:]
            self.buffer_y = self.buffer_y[-self.max_size:]

    def sample(self, n):
        """Obtiene n muestras aleatorias del buffer."""
        idx = np.random.choice(len(self.buffer_X), size=min(n, len(self.buffer_X)), replace=False)
        return (np.array([self.buffer_X[i] for i in idx]),
                np.array([self.buffer_y[i] for i in idx]))

    def __len__(self):
        return len(self.buffer_X)


def incremental_train(model, X_new, y_new, replay_buffer, device,
                      epochs=5, lr=1e-4, batch_size=64, replay_ratio=0.5):
    """
    Actualiza el modelo con nuevos datos usando replay buffer.

    Combina nuevos datos con muestras del buffer para evitar la
    perdida catastrofica. Usa learning rate reducido para preservar
    el conocimiento previo.

    Parameters
    ----------
    model : ECG_CNN
        Modelo a actualizar.
    X_new : np.ndarray
        Nuevos latidos ECG.
    y_new : np.ndarray
        Etiquetas de los nuevos latidos.
    replay_buffer : ReplayBuffer
        Buffer con muestras anteriores.
    device : torch.device
        CPU o GPU.
    epochs : int
        Epocas de reentrenamiento (pocas para no sobreajustar).
    lr : float
        Learning rate reducido para preservar conocimiento previo.
    replay_ratio : float
        Proporcion de muestras del buffer vs nuevas.

    Returns
    -------
    float
        Loss promedio del reentrenamiento.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.NLLLoss()
    model.train()

    total_loss = 0.0
    n_batches  = 0

    # Agregar nuevos datos al buffer
    replay_buffer.add(X_new, y_new)

    for epoch in range(epochs):
        # Mezclar nuevos datos con muestras del buffer
        n_replay = int(len(X_new) * replay_ratio)

        if len(replay_buffer) > 0 and n_replay > 0:
            X_replay, y_replay = replay_buffer.sample(n_replay)
            X_combined = np.concatenate([X_new, X_replay], axis=0)
            y_combined = np.concatenate([y_new, y_replay], axis=0)
        else:
            X_combined = X_new
            y_combined = y_new

        # Mezclar aleatoriamente
        perm = np.random.permutation(len(X_combined))
        X_combined = X_combined[perm]
        y_combined = y_combined[perm]

        X_t = torch.tensor(X_combined, dtype=torch.float32).unsqueeze(1)
        y_t = torch.tensor(y_combined, dtype=torch.long)
        loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1

    return total_loss / n_batches if n_batches > 0 else 0.0


print("Modulo de reentrenamiento continuo definido.")
print("  ReplayBuffer     : buffer de muestras anteriores")
print("  incremental_train: actualiza modelo sin reiniciar")

## Paso 4: Simulacion de Reentrenamiento Continuo

Partiendo del modelo base, procesamos cada lote de forma secuencial simulando un despliegue clinico real donde los datos de nuevos pacientes llegan periodicamente. Para cada lote nuevo se sigue este protocolo:

1. **Evaluar ANTES:** medir el rendimiento actual del modelo adaptativo en el nuevo lote.
2. **Reentrenar:** aplicar `incremental_train` con el replay buffer (mezcla nuevos datos + memoria del pasado).
3. **Evaluar DESPUES:** medir el rendimiento tras la actualizacion.

El learning rate reducido (`lr=1e-4`) y el numero limitado de epocas (`epochs=5`) evitan sobreajustar a los nuevos datos y preservan el conocimiento general del modelo.

In [ ]:
# Crear copia del modelo base para el modelo adaptativo
adaptive_model = copy.deepcopy(base_model)
replay_buffer  = ReplayBuffer(max_size=2000)

# Inicializar buffer con el primer lote (datos de entrenamiento original)
X0, y0 = batches[0]
replay_buffer.add(X0, y0)

adaptive_accuracies = []
adaptive_f1s        = []
retrain_losses      = []

print("Simulacion de reentrenamiento continuo:")
print("{:>6} | {:>12} | {:>10} | {:>10} | {:>10}".format(
    "Lote", "Acc Antes", "Acc Despues", "F1 Despues", "Loss Retrain"))
print("-" * 60)

for i, (bx, by) in enumerate(batches):
    # Evaluar ANTES del reentrenamiento (rendimiento actual)
    acc_before, _ = evaluate_batch(adaptive_model, bx, by, device)

    # Reentrenar con el nuevo lote (excepto el primero)
    if i > 0:
        loss = incremental_train(adaptive_model, bx, by, replay_buffer, device,
                                 epochs=5, lr=1e-4)
    else:
        loss = 0.0

    # Evaluar DESPUES del reentrenamiento
    acc_after, f1_after = evaluate_batch(adaptive_model, bx, by, device)

    adaptive_accuracies.append(acc_after)
    adaptive_f1s.append(f1_after)
    retrain_losses.append(loss)

    print("{:>6} | {:>11.2f}% | {:>10.2f}% | {:>10.4f} | {:>10.4f}".format(
        i, acc_before*100, acc_after*100, f1_after, loss))

print("\nReentrenamiento continuo completado.")

In [ ]:
# Comparacion: modelo estatico vs modelo adaptativo
batches_range = range(N_BATCHES)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(batches_range, [a*100 for a in static_accuracies],
         "o--", color="tomato",    label="Modelo Estatico",   linewidth=2)
ax1.plot(batches_range, [a*100 for a in adaptive_accuracies],
         "o-",  color="steelblue", label="Modelo Adaptativo", linewidth=2)
ax1.set_title("Accuracy por Lote")
ax1.set_xlabel("Lote (tiempo)")
ax1.set_ylabel("Accuracy (%)")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(list(batches_range))

ax2.plot(batches_range, static_f1s,
         "o--", color="tomato",    label="Modelo Estatico",   linewidth=2)
ax2.plot(batches_range, adaptive_f1s,
         "o-",  color="steelblue", label="Modelo Adaptativo", linewidth=2)
ax2.set_title("F1-Score Macro por Lote")
ax2.set_xlabel("Lote (tiempo)")
ax2.set_ylabel("F1 Macro")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(list(batches_range))

plt.suptitle("Modelo Estatico vs Modelo Adaptativo (Reentrenamiento Continuo)", fontsize=12)
plt.tight_layout()
plt.savefig(DOCS_PATH + "/fase3_comparacion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada.")

## Paso 5: Evaluacion Final Comparativa

In [ ]:
# Evaluacion final: modelo estatico vs adaptativo en todos los datos
X_all = np.concatenate([b[0] for b in batches[1:]], axis=0)  # lotes 1-4
y_all = np.concatenate([b[1] for b in batches[1:]], axis=0)

def full_evaluate(model, X_data, y_data, device, batch_size=256):
    """Evaluacion completa con todas las metricas."""
    model.eval()
    X_t = torch.tensor(X_data, dtype=torch.float32).unsqueeze(1)
    y_t = torch.tensor(y_data, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=False)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(device))
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(yb.numpy())

    return np.array(all_preds), np.array(all_labels)

preds_static,   labels_all = full_evaluate(base_model,     X_all, y_all, device)
preds_adaptive, _          = full_evaluate(adaptive_model, X_all, y_all, device)

acc_s  = accuracy_score(labels_all, preds_static)
acc_a  = accuracy_score(labels_all, preds_adaptive)
f1_s   = f1_score(labels_all, preds_static,   average="macro", zero_division=0)
f1_a   = f1_score(labels_all, preds_adaptive, average="macro", zero_division=0)
f1ws   = f1_score(labels_all, preds_static,   average="weighted", zero_division=0)
f1wa   = f1_score(labels_all, preds_adaptive, average="weighted", zero_division=0)

print("=" * 55)
print("  COMPARACION FINAL: Estatico vs Adaptativo")
print("=" * 55)
print("{:<20} {:>15} {:>15}".format("Metrica", "Estatico", "Adaptativo"))
print("-" * 55)
print("{:<20} {:>14.2f}% {:>14.2f}%".format("Accuracy",     acc_s*100, acc_a*100))
print("{:<20} {:>15.4f} {:>15.4f}".format("F1 Macro",      f1_s,  f1_a))
print("{:<20} {:>15.4f} {:>15.4f}".format("F1 Weighted",   f1ws,  f1wa))
print()
print("Mejora en Accuracy : {:.2f}%".format((acc_a - acc_s)*100))
print("Mejora en F1 Macro : {:.4f}".format(f1_a - f1_s))

In [ ]:
# Guardar modelo adaptativo
adaptive_path = MODELS_PATH + "/ADAPT-ECG-RETRAINED.pth"
torch.save(adaptive_model.state_dict(), adaptive_path)

# Guardar resultados de comparacion
results = {
    "modelo_estatico": {
        "accuracy":    round(float(acc_s), 4),
        "f1_macro":    round(float(f1_s),  4),
        "f1_weighted": round(float(f1ws),  4),
    },
    "modelo_adaptativo": {
        "accuracy":    round(float(acc_a), 4),
        "f1_macro":    round(float(f1_a),  4),
        "f1_weighted": round(float(f1wa),  4),
    },
    "mejora": {
        "accuracy": round(float((acc_a - acc_s)*100), 2),
        "f1_macro": round(float(f1_a - f1_s), 4),
    },
    "replay_buffer_size": 2000,
    "n_batches":          N_BATCHES,
    "epochs_per_batch":   5,
    "lr_retraining":      1e-4,
}

results_path = MODELS_PATH + "/fase3_resultados.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Modelo adaptativo guardado : " + adaptive_path)
print("Resultados guardados       : " + results_path)
print()
print("Accuracy  Estatico  : {:.2f}%".format(acc_s*100))
print("Accuracy  Adaptativo: {:.2f}%".format(acc_a*100))
print("F1 Macro  Estatico  : {:.4f}".format(f1_s))
print("F1 Macro  Adaptativo: {:.4f}".format(f1_a))